# Volume 6, Chapter 6: Measures (Python companion)

Exercises first, solutions at the end. Measures on a computer: digit statistics of $\pi$, $e$, $\sqrt2$ and Champernowne's number, a famous bad random number generator caught by a test on triples, many independent uniforms dealt from one, sampling the singular Cantor law, fat Cantor sets and $C+C$, and the slow normality of Champernowne's sequence. Requirements: numpy, scipy, mpmath and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P6.1 (★) Are $\pi$, $e$ and $\sqrt2$ normal?** Compute $100\,000$ decimal digits of each. Count single digits and pairs of digits, and run a chi-square test against the uniform frequencies of Borel's theorem (Theorem 6.19). Do the same for Champernowne's number $0.123456789101112\ldots$, and comment.

**P6.2 (★★) CS application: testing random number generators.** IBM's generator RANDU (1960s) is $x_{n+1}=65539\,x_n\bmod2^{31}$, $u_n=x_n/2^{31}$. (a) Show that $x_{n+2}=6x_{n+1}-9x_n\bmod2^{31}$. (b) Plot consecutive triples $(u_n,u_{n+1},u_{n+2})$ and find the planes. (c) Design a chi-square test on triples that RANDU fails and numpy's PCG64 passes. Why can single-digit frequencies never detect this flaw?

**P6.3 (★) One number, many numbers.** Draw one uniform number with $4000$ random bits, deal its digits as in Theorem 6.17 into $U_1,\dots,U_6$, and compute their sample correlations over $10\,000$ repetitions. Then turn $U_1,U_2$ into two independent standard Gaussians with the quantile transform of Proposition 6.14.

**P6.4 (★) Sampling a singular law.** Sample the Cantor law $\mu_c$ (the law of $\sum_k2B_k3^{-k}$ with fair bits $B_k$), plot its empirical distribution function against the Cantor staircase, and check that all samples lie in $C_n$ for $n=10$. Then sample the law with distribution function $F(t)=\frac t2$ on $[0,1[$, $F=1$ on $[1,\infty[$ (an atom at $1$) with the quantile transform.

**P6.5 (★★) Fat and thin Cantor sets.** Compute the measure of the stages of the fat Cantor set of Exercise 6.2 and the length of its largest remaining interval. Check numerically that $C+C=[0,2]$ (Exercise 6.3): for $1000$ random $z\in[0,2]$, find $x,y\in C$ with $x+y=z$ to $30$ ternary digits.

**P6.6 (★★) How fast do frequencies converge?** For a random sequence of bits, $\frac{N_n}n-\frac12$ is of order $\frac1{2\sqrt n}$ (central limit theorem). For Champernowne's binary sequence $1\,10\,11\,100\ldots$, compute $\frac{N_n}n-\frac12$ for $n$ up to $10^7$ and compare. Is the binary Champernowne sequence normal?

## Solutions

### P6.1

In [ ]:
from mpmath import mp
from collections import Counter
from scipy.stats import chisquare
N = 100_000
mp.dps = N + 20

sources = {"pi": str(mp.pi).replace(".", "")[1:N + 1], "e": str(mp.e).replace(".", "")[1:N + 1],
           "sqrt 2": str(mp.sqrt(2)).replace(".", "")[1:N + 1]}
champ = "".join(str(k) for k in range(1, 30000))[:N]
sources["Champernowne"] = champ
for name, s in sources.items():
    c1 = Counter(s); c2 = Counter(s[i:i + 2] for i in range(0, N - 1, 2))
    p1 = chisquare([c1[str(d)] for d in range(10)]).pvalue
    p2 = chisquare([c2[f"{d:02d}"] for d in range(100)]).pvalue
    print(f"{name:13s} digit counts {[c1[str(d)] for d in range(10)]}  p-values: digits {p1:.3f}, pairs {p2:.3f}")

For $\pi$, $e$ and $\sqrt2$ the counts fluctuate by a few hundred around $10\,000$, about $\sqrt{10\,000\cdot0.9}\approx95$ per digit at one standard deviation, and the $p$-values are scattered between $0.02$ and $0.9$: with eight tests, a couple of values near $0.03$ are what chance produces. Nothing distinguishes these digits from random ones. That is evidence, not proof: no one can prove that the digit $7$ occurs infinitely often in $\pi$. Champernowne's number is *provably* normal in base $10$, yet its first $100\,000$ digits fail the test spectacularly: up to $n\approx10^5$ we are still among the $5$-digit integers, whose leading digits are never $0$ and lean towards small values. Normality is an asymptotic property and says nothing about any finite prefix; a statistical test sees only prefixes.

### P6.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chisquare

def randu(n, seed=1):
    x = np.empty(n, dtype=np.int64); x[0] = seed
    for i in range(1, n):
        x[i] = (65539 * x[i - 1]) % 2 ** 31
    return x

x = randu(30_000)
print("x[n+2] - 6x[n+1] + 9x[n] mod 2^31 == 0:", np.all((x[2:] - 6 * x[1:-1] + 9 * x[:-2]) % 2 ** 31 == 0))
u = x / 2 ** 31
t = np.stack([u[:-2], u[1:-1], u[2:]], axis=1)[::3]
c = 9 * t[:, 0] - 6 * t[:, 1] + t[:, 2]
print("9u_n - 6u_n+1 + u_n+2 takes only the integer values", sorted(int(v) for v in set(np.round(c).astype(int))))

ax = plt.figure(figsize=(5, 5)).add_subplot(projection="3d")
ax.scatter(*t[:4000].T, s=1); ax.view_init(elev=10, azim=-58); ax.set_title("RANDU triples"); plt.show()

def triple_test(u, k=10):
    t = np.floor(np.stack([u[:-2], u[1:-1], u[2:]], axis=1)[::3] * k).astype(int)
    cells = t[:, 0] * k * k + t[:, 1] * k + t[:, 2]
    return chisquare(np.bincount(cells, minlength=k ** 3)).pvalue
print("triple test p-value, RANDU:", triple_test(randu(300_000) / 2 ** 31))
print("triple test p-value, PCG64:", triple_test(np.random.default_rng(0).random(300_000)))

(a) $65539=2^{16}+3$, so $65539^2=2^{32}+6\cdot2^{16}+9\equiv6(2^{16}+3)-9\pmod{2^{31}}$: hence $x_{n+2}\equiv6x_{n+1}-9x_n$. (b) Dividing by $2^{31}$, $9u_n-6u_{n+1}+u_{n+2}$ is an integer, and since each $u\in[0,1[$ it lies between $-5$ and $9$: all triples lie on $15$ parallel planes. Seen from the right angle, the "random" cube is a stack of sheets. (c) Cut the cube into $1000$ cells: the points crowd onto the planes, so some cells are overfull and others nearly empty, and the chi-square $p$-value of RANDU is about $10^{-33}$, while PCG64 passes. Single values $u_n$ of RANDU are well distributed: the flaw is in the *joint* law of consecutive values, that is, in independence, not in uniformity. Testing a generator means testing finite-dimensional joint laws, exactly the events of Theorem 6.17; Marsaglia's 1968 paper "Random numbers fall mainly in the planes" showed that every linear congruential generator has this defect in some dimension.

### P6.3

In [ ]:
import numpy as np
rng = np.random.default_rng(7)
K, J, reps = 6, 40, 10_000            # 6 numbers, 40 bits each: positions up to 2^5 * 79 < 4000
U = np.zeros((reps, K))
for r in range(reps):
    bits = rng.integers(0, 2, 4000)   # D_1 ... D_4000 of one uniform omega
    for k in range(1, K + 1):
        pos = [2 ** (k - 1) * (2 * j - 1) for j in range(1, J + 1)]
        U[r, k - 1] = sum(bits[p - 1] * 2.0 ** -j for j, p in enumerate(pos, start=1))
print("means:", U.mean(0).round(3))
print("correlation matrix:\n", np.corrcoef(U.T).round(3))

from scipy.stats import norm, kstest
Z = norm.ppf(U[:, :2])                 # quantile transform G = F^{-1}
print("KS test of Z1 against N(0,1): p =", round(kstest(Z[:, 0], "norm").pvalue, 3), "  corr(Z1, Z2) =", round(np.corrcoef(Z.T)[0, 1], 3))

The means are close to $\frac12$ and the off-diagonal correlations are of order $\frac1{\sqrt{10\,000}}=0.01$, as for independent variables. Correlation zero is much weaker than independence, but here independence holds exactly by Theorem 6.17. With $40$ bits each, the $U_k$ are uniform up to $2^{-40}$; the quantile transform of the Gaussian law then produces Gaussians (the Kolmogorov–Smirnov $p$-value is unremarkable). In practice, generators do the same: a stream of random bits is cut into words, and every distribution is obtained from uniforms by a quantile transform or a cleverer equivalent (Box–Muller, Volume 3).

### P6.4

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(3)
n, K = 20_000, 40
B = rng.integers(0, 2, (n, K))
X = (2 * B * 3.0 ** -np.arange(1, K + 1)).sum(1)

def cantor_staircase(x, depth=40):
    y, scale = 0.0, 0.5
    for _ in range(depth):
        x *= 3
        if x < 1: pass
        elif x < 2: return y + scale
        else: y += scale; x -= 2
        scale /= 2
    return y

ts = np.linspace(0, 1, 801)
plt.plot(ts, [np.mean(X <= t) for t in ts], label="empirical")
plt.plot(ts, [cantor_staircase(t) for t in ts], "--", label="Cantor staircase")
plt.legend(); plt.show()

def in_Cn(x, n=10):
    for _ in range(n):
        x *= 3; d = int(x); x -= d
        if d == 1: return False
    return True
print("fraction of samples in C_10:", np.mean([in_Cn(x) for x in X]))

U = rng.random(100_000)
G = np.where(U <= 0.5, 2 * U, 1.0)          # quantile of F(t) = t/2 on [0,1), F = 1 on [1, inf)
print("P(X = 1) ~", np.mean(G == 1.0), "  P(X <= 0.4) ~", np.mean(G <= 0.4), "(exact 0.2)")

The empirical distribution function of the samples hugs the Cantor staircase: $\mu_c$ is the Lebesgue–Stieltjes measure of the staircase, a continuous law (no atoms) that gives measure $1$ to the null set $C$. Every sample lies in $C_{10}$ (its ternary digits are $0$ or $2$ by construction, so up to rounding it lies in every $C_n$). For the mixed law, the quantile function is $G(u)=2u$ for $u\le\frac12$ and $G(u)=1$ for $u>\frac12$: the flat part of $G$ is the atom, of mass $F(1)-F(1^-)=\frac12$ (Exercise 6.1).

### P6.5

In [ ]:
import numpy as np
from fractions import Fraction
removed, ell = Fraction(0), Fraction(1)
for n in range(1, 11):
    removed += Fraction(2 ** (n - 1), 4 ** n)
    ell = (ell - Fraction(1, 4 ** n)) / 2
    print(f"step {n:2d}: measure left {float(1 - removed):.6f}, longest interval {float(ell):.2e}  (2^-n = {2.0**-n:.2e})")

rng = np.random.default_rng(0)
def ternary(z, D=30):
    d = []
    for _ in range(D):
        z *= 3; k = min(int(z), 2); d.append(k); z -= k
    return d
worst = 0.0
for _ in range(1000):
    z = rng.random() * 2
    t = ternary(z / 2)
    u = [1 if k >= 1 else 0 for k in t]; v = [1 if k == 2 else 0 for k in t]
    x = sum(2 * a * 3.0 ** -(j + 1) for j, a in enumerate(u)); y = sum(2 * b * 3.0 ** -(j + 1) for j, b in enumerate(v))
    worst = max(worst, abs(x + y - z))
print("largest |x + y - z| over 1000 trials:", worst, " (3^-30 =", 3.0 ** -30, ")")

The fat Cantor set keeps measure $\frac12+2^{-n-1}$ after step $n$, tending to $\frac12$, while its longest interval shrinks like $2^{-n}$ (it stays between $2^{-n-1}$ and $2^{-n}$, as in the solution of Exercise 6.2): a set of measure $\frac12$ containing no interval. For $C+C$, splitting each ternary digit $t=u+v$ of $\frac z2$ with $u,v\in\{0,1\}$ and doubling gives $x,y\in C$ with $x+y=z$ up to $2\cdot3^{-30}$, the truncation of the $30$-digit expansions (and floating point). A null set plus itself fills an interval.

### P6.6

In [ ]:
import numpy as np
M = 10 ** 7
bits = np.zeros(M, dtype=np.uint8); pos = 0; k = 1
while pos < M:
    b = np.frombuffer(bin(k)[2:].encode(), dtype=np.uint8) - 48
    take = min(len(b), M - pos); bits[pos:pos + take] = b[:take]; pos += take; k += 1
rand = np.random.default_rng(0).integers(0, 2, M)
for n in [10 ** 3, 10 ** 4, 10 ** 5, 10 ** 6, 10 ** 7]:
    dc = bits[:n].mean() - 0.5; dr = rand[:n].mean() - 0.5
    print(f"n=1e{int(np.log10(n))}: Champernowne {dc:+.5f}   random {dr:+.5f}   1/(2 sqrt n) = {0.5/np.sqrt(n):.5f}   1/log2(n) = {1/np.log2(n):.4f}")

The random sequence's excess frequency of ones is of order $\frac1{2\sqrt n}$, as the central limit theorem predicts. Champernowne's binary sequence has a positive bias that decays only like $\frac1{\log n}$: every binary integer starts with a $1$, and at scale $n$ the integers written have about $\log_2n$ bits, one of which is a forced leading $1$. The bias does tend to $0$, and in fact the binary Champernowne sequence is normal in base $2$ (Champernowne's argument works in every base), but extremely slowly. Normality allows any $o(n)$ error; random sequences are much better than merely normal.